# Analyzing Wikipedia Pages

In this project, we will analyze Wikipedia pages to gain insights into the content and structure of the articles. We will use Python and various libraries to scrape and process the data, and then visualize the results to better understand the Wikipedia ecosystem. We will implement a quick script command to collect 200 MB worth of data for our processing. For reference, the below script was generated by Claude AI as the course hasn't reached webscraping as of yet. Using the script, we generated 6648 files, which are otherwise found in the `wiki` folder in this directory.

In [20]:
import os, time, requests
from urllib.parse import quote

# --- settings ---
OUT_DIR   = 'wiki'
TARGET_MB = 1000      # stop once the folder reaches this size
DELAY     = 0.3      # seconds between requests
# ----------------

HEADERS = {'User-Agent': 'grep-practice/1.0 (learning project)'}
API = 'https://en.wikipedia.org/w/api.php'
os.makedirs(OUT_DIR, exist_ok=True)

def folder_bytes():
    return sum(os.path.getsize(os.path.join(OUT_DIR, f))
               for f in os.listdir(OUT_DIR))

def random_titles(n=50):
    r = requests.get(API, headers=HEADERS, timeout=30, params={
        'action': 'query', 'list': 'random',
        'rnnamespace': 0, 'rnlimit': n, 'format': 'json'})
    r.raise_for_status()
    return [p['title'] for p in r.json()['query']['random']]

target = TARGET_MB * 1_000_000
total  = folder_bytes()
new    = 0
print(f'starting at {total/1e6:.1f} MB, target {TARGET_MB} MB')

try:
    while total < target:
        for title in random_titles():
            if total >= target:
                break
            slug = quote(title.replace(' ', '_'), safe='')
            path = os.path.join(OUT_DIR, slug + '.html')
            if os.path.exists(path):
                continue
            try:
                r = requests.get(f'https://en.wikipedia.org/wiki/{slug}',
                                 headers=HEADERS, timeout=30)
                if r.status_code != 200:
                    continue
            except requests.RequestException:
                continue
            with open(path, 'w', encoding='utf-8') as f:
                f.write(r.text)
            total += os.path.getsize(path)
            new += 1
            if new % 25 == 0:
                print(f'{total/1e6:6.1f} MB   {new} new files', end='\r')
            time.sleep(DELAY)
except KeyboardInterrupt:
    print('\ninterrupted')

print(f'\ndone: {len(os.listdir(OUT_DIR))} files, {folder_bytes()/1e6:.1f} MB')

starting at 295.9 MB, target 1000 MB
1000.0 MB   4650 new files
done: 6648 files, 1000.0 MB


We can now begin analyzing the collected Wikipedia pages to gain insights into the content and structure of the articles. We will use Python and various libraries to process the data, and then visualize the results to better understand the Wikipedia ecosystem. The first segment of the task is to check if we can actually read the contents of the folder, and if we can ready any of the files.

In [17]:
import os
files = os.listdir("wiki")

print(f"---- File Directory -----\n"
      f"The number of files in this directory is {len(files)}, which is too many to show. We will show the first 10 of {len(files)} files, and the sample content of the first file.\n")

for i in range(10):
    print(f"{i+1}. {files[i]}")

print(f"\n"
      f"---- Sample File ----\n"
      f"Here, we will look at the contents of the first file. (First 10 lines)\n")

with open(os.path.join("wiki", files[0]), encoding = "UTF-8") as file:
    lines = file.readlines()

for line in lines[:10]:
    print(line, end="")

---- File Directory -----
The number of files in this directory is 1359, which is too many to show. We will show the first 10 of 1359 files, and the sample content of the first file.

1. %C3%81ngel_Garc%C3%ADa_%28basketball%2C_born_1988%29.html
2. %C3%81ngel_Malvicino.html
3. %C3%87eltikli%2C_Bismil.html
4. %C3%89milie_Gamelin.html
5. %C3%9Apohlavy.html
6. %C3%9Eorger%C3%B0ur_Anna_Atlad%C3%B3ttir.html
7. %C5%81%C4%85cko%2C_West_Pomeranian_Voivodeship.html
8. %C5%BB%C3%B3%C5%82cin.html
9. %E1%B9%A2.html
10. .gc.ca.html

---- Sample File ----
Here, we will look at the contents of the first file. (First 10 lines)

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-conten

## Multiprocessing

We will use Python's multiprocessing module to speed up the analysis process. This will allow us to process multiple files simultaneously, significantly reducing the time required to analyze the entire dataset. To start of with, lets get a total line count using both single and multiprocessing.

In [18]:
# Single Process

import time

sgl_processing_lines = []

start = time.time()
for file in files:
    with open(os.path.join("wiki", file), encoding="UTF-8") as file:
        sgl_processing_lines.append(len(file.readlines()))
end = time.time()
sgl_processing_time = end - start

print(f"Counted {sum(sgl_processing_lines):,} lines in a single process in {sgl_processing_time:.3f}s")

Counted 1,393,443 lines in a single process in 64.615s


In [26]:
# Multi Processing

%%writefile wiki_mapreduce.py

import os

def mapper(file_chunk):
    line_count = []
    for file in file_chunk:
        with open(os.path.join("wiki", file), encoding="UTF-8") as f:
            line_count.append(len(f.readlines()))
    return sum(line_count), len(file_chunk)

def reducer(a, b):
    return a + b

Writing wiki_mapreduce.py


In [ ]:
import os
import time
import math
import functools
import importlib
from multiprocessing import Pool

import wiki_mapreduce
importlib.reload(wiki_mapreduce)          # picks up edits without a kernel restart
from wiki_mapreduce import mapper, reducer

def make_chunks(data, num_chunks):
    chunk_size = math.ceil(len(data) / num_chunks)
    return [data[i:i+chunk_size] for i in range(0, len(data), chunk_size)]

def mapper(file_chunk):
    line_count = []
    for file in file_chunk:
        with open(os.path.join("wiki", file), encoding="UTF-8") as f:
            line_count.append(len(f.readlines()))
    print("At least one done.")
    return sum(line_count)

def reducer(a, b):
    return a + b

def map_reduce(data, num_processes, mapper, reducer):
    chunks = make_chunks(data, num_processes)
    pool = Pool(num_processes)
    chunk_results = pool.map(mapper, chunks)
    return functools.reduce(reducer, chunk_results)

if __name__ == "__main__":
    workers = (os.cpu_count() // 3) * 2
    start = time.time()
    multi_processing_lines = map_reduce(files, workers, mapper, reducer)
    end = time.time()
    multi_processing_time = end - start

    print(f"Counted {multi_processing_lines:,} lines in a multi-process in {multi_processing_time:.3f}s")
